In [5]:
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

In [2]:
Ytrain  = pd.read_pickle('../data/ytrain.pkl')
Ytest   = pd.read_pickle('../data/ytest.pkl')

Xtrain  = pd.read_pickle('../data/xtrain.pkl')
Xtest   = pd.read_pickle('../data/xtest.pkl')

In [17]:
Xtrain_embedded = PCA(n_components=15).fit_transform(Xtrain)
Xtest_embedded =  PCA(n_components=15).fit_transform(Xtest)

In [18]:
dt_classificador = DecisionTreeClassifier(criterion='entropy', max_depth=None, random_state=42)
rf = RandomForestClassifier(n_estimators= 100, criterion='entropy')
svm = SVC(kernel='poly', degree=4)
nb = GaussianNB()

classificadores = [dt_classificador, rf, svm, nb]
nomeClassificadores = ['Decision Tree', 'Random Forest', 'SVM', 'Naive Bayes']

In [23]:
for i, classificador in enumerate(classificadores):
    classificador.fit(Xtrain_embedded, Ytrain)

    #print(nomeClassificadores[i])
    

    escores_cv = cross_val_score(classificador, Xtrain_embedded, Ytrain, cv=5)
    print(f'{nomeClassificadores[i]} & {str(escores_cv.round(4)).replace(' ', '&')} & {np.mean(escores_cv).round(4)} & {accuracy_score(ypred, Ytest)}')

    ypred = classificador.predict(Xtest_embedded)
    
    #print(classification_report(Ytest, ypred))

    

Decision Tree & [0.8407&0.8616&0.846&&0.8613&0.911&] & 0.8641 & 0.7793427230046949
Random Forest & [0.9138&0.9217&0.9034&0.9005&0.9188] & 0.9117 & 0.7699530516431925
SVM & [0.8851&0.8877&0.8642&0.8455&0.8874] & 0.874 & 0.8450704225352113
Naive Bayes & [0.8146&0.8251&0.7911&0.8063&0.8298] & 0.8134 & 0.8028169014084507


In [44]:
import numpy as np

from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.metrics import classification_report

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# ==========================================================
# Modelos e distribuições dos hiperparâmetros
# ==========================================================

modelos = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {
            "criterion": ["gini", "entropy"],
            "max_depth": [None] + list(range(2, 31)),
            "min_samples_split": range(2, 21),
            "min_samples_leaf": range(1, 11)
        }
    ),

    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators": np.arange(50, 501, 50),
            "criterion": ["gini", "entropy"],
            "max_depth": [None] + list(range(5, 31, 5)),
            "min_samples_split": range(2, 11),
            "min_samples_leaf": range(1, 6),
            "max_features": ["sqrt", "log2", None]
        }
    ),

    "SVM": (
        SVC(),
        {
            "C": np.logspace(-2, 3, 10),
            "kernel": ["linear", "rbf", "poly"],
            "gamma": np.logspace(-4, 1, 10),
            "degree": [2, 3, 4, 5]
        }
    ),

    "Naive Bayes": (
        GaussianNB(),
        {
            "var_smoothing": np.logspace(-12, -6, 100)
        }
    )
}

# ==========================================================
# Busca aleatória
# ==========================================================

for nome, (modelo, parametros) in modelos.items():

    print("=" * 80)
    print(nome)

    busca = RandomizedSearchCV(
        estimator=modelo,
        param_distributions=parametros,
        n_iter=30,              # número de combinações testadas
        scoring="accuracy",
        cv=5,
        random_state=42,
        n_jobs=4,
        verbose=1
    )

    busca.fit(Xtrain_embedded, Ytrain)

    melhor_modelo = busca.best_estimator_

    print("\nMelhor classificador:")
    print(melhor_modelo)

    print("\nMelhores hiperparâmetros:")
    print(busca.best_params_)

    print(f"\nMelhor acurácia média (5-fold): {busca.best_score_:.4f}")

    # Validação cruzada do melhor modelo
    scores = cross_val_score(
        melhor_modelo,
        Xtrain_embedded,
        Ytrain,
        cv=5,
        scoring="accuracy"
    )

    print("\nValidação Cruzada (5 folds)")
    print("Escores:", np.round(scores, 4))
    print(f"Média: {scores.mean():.4f}")
    print(f"Desvio padrão: {scores.std():.4f}")

    # Teste
    ypred = melhor_modelo.predict(Xtest_embedded)

    print("\nResultado no conjunto de teste")
    print(classification_report(Ytest, ypred))

    print("\n")

Decision Tree
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Melhor classificador:
DecisionTreeClassifier(criterion='entropy', max_depth=30, min_samples_leaf=4,
                       min_samples_split=19, random_state=42)

Melhores hiperparâmetros:
{'min_samples_split': 19, 'min_samples_leaf': 4, 'max_depth': 30, 'criterion': 'entropy'}

Melhor acurácia média (5-fold): 0.8787

Validação Cruzada (5 folds)
Escores: [0.8486 0.893  0.8695 0.8901 0.8927]
Média: 0.8787
Desvio padrão: 0.0174

Resultado no conjunto de teste
              precision    recall  f1-score   support

           1       0.90      0.91      0.90       159
           2       0.53      0.50      0.51        36
           3       0.53      0.50      0.51        18

    accuracy                           0.81       213
   macro avg       0.65      0.64      0.64       213
weighted avg       0.80      0.81      0.80       213



Random Forest
Fitting 5 folds for each of 30 candidates, totalling 150 fits

M

KeyboardInterrupt: 